In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

plt.rcParams.update({"font.family": "sans-serif", "font.sans-serif": ["Arial", "Helvetica", "DejaVu Sans"],
                     "pdf.fonttype": 42, "svg.fonttype": "none", "figure.dpi": 110})

ROI = "HIP"
CHEMS = None
SLIM_DIR = "/Volumes/Lexar/mglurs-brain-sc-datasets/derived/regions"
FIG_DIR = "figures/allen_wmb"
MIN_CELLS = 200
GRMS = [f"Grm{i}" for i in range(1, 9)]

df = pd.read_parquet(f"{SLIM_DIR}/{ROI}_slim.parquet")
if CHEMS:
    df = df[df["chem"].isin(CHEMS)]
genes = [g for g in GRMS if g in df.columns]
cls_num = pd.to_numeric(df["class"].astype(str).str[:2], errors="coerce").fillna(0)
df["neuron"] = ~cls_num.between(30, 34)
os.makedirs(FIG_DIR, exist_ok=True)
tag = ROI + ("_" + "-".join(c.replace("WMB-10X", "") for c in CHEMS) if CHEMS else "")
print(ROI, len(df), "cells;", df["subclass"].nunique(), "subclasses; chem:", df["chem"].value_counts().to_dict())

from scipy.stats import mannwhitneyu
rng = np.random.default_rng(0)
neu = df[df["neuron"]]
sizes = neu["subclass"].value_counts()
focal = sizes[sizes >= MIN_CELLS].index.tolist()
rows = []
for g in genes:
    lin = neu[g].to_numpy()
    cpm = np.expm1(lin)
    sub = neu["subclass"].to_numpy()
    for s in focal:
        m = sub == s
        if (~m).sum() < MIN_CELLS:
            continue
        a = lin[m]
        b = lin[~m]
        bs = rng.choice(b, size=min(20000, len(b)), replace=False)
        p = mannwhitneyu(a, bs, alternative="greater", method="asymptotic").pvalue if np.ptp(np.r_[a, bs]) > 0 else 1.0
        rows.append(dict(roi=ROI, gene=g, subclass=s, cls=neu.loc[m, "class"].iloc[0], n_cells=int(m.sum()),
                         mean_cpm=cpm[m].mean(), mean_cpm_other_neurons=cpm[~m].mean(),
                         fold=(cpm[m].mean() + 1) / (cpm[~m].mean() + 1), median_log2=np.median(a),
                         median_log2_other_neurons=np.median(b), p_greater=p))
res = pd.DataFrame(rows)
def bh(p):
    p = np.asarray(p)
    o = np.argsort(p)
    q = np.empty_like(p, dtype=float)
    q[o] = np.minimum.accumulate((p[o] * len(p) / (np.arange(len(p)) + 1))[::-1])[::-1]
    return np.minimum(q, 1)
res["q_greater"] = res.groupby("gene")["p_greater"].transform(bh)
res = res.sort_values(["gene", "fold"], ascending=[True, False])
os.makedirs("results/allen_wmb/enrichment", exist_ok=True)
res.to_csv(f"results/allen_wmb/enrichment/{tag}_enrichment.csv", index=False)
print("fold = (mean CPM in subclass + 1) / (mean CPM in all other neuronal cells in region + 1); p = one-sided rank-sum vs other neurons")
for g in genes:
    print(res[res["gene"] == g].head(6)[["gene", "subclass", "n_cells", "mean_cpm", "mean_cpm_other_neurons", "fold", "q_greater"]].round(3).to_string(index=False))